In [14]:
# =========================
# 1. Install (run once in Colab)
# =========================
# !pip install torch torchvision matplotlib pillow -q


# =========================
# 2. Imports
# =========================
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision import transforms as T
import matplotlib.pyplot as plt
from PIL import Image
import requests

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [1]:
data = [
    ("ich bin ein student", "i am a student"),
    ("ich liebe ki", "i love ai"),
    ("er ist klug", "he is smart"),
    ("sie ist glücklich", "she is happy"),
    ("ich lerne", "i am learning"),
    ("das ist gut", "that is good"),
    ("das ist schlecht", "that is bad"),
    ("ich gehe nach hause", "i go home"),
    ("er spielt fussball", "he plays football"),
    ("sie liest ein buch", "she reads a book"),
    ("ich trinke wasser", "i drink water"),
    ("wir gehen zur schule", "we go to school"),
    ("sie kocht essen", "she cooks food"),
    ("ich arbeite heute", "i work today"),
    ("er fährt ein auto", "he drives a car"),
    ("ich sehe dich", "i see you"),
    ("wir lernen zusammen", "we learn together"),
    ("das ist mein freund", "that is my friend"),
    ("ich habe hunger", "i am hungry"),
    ("ich habe durst", "i am thirsty"),
    ("er ist müde", "he is tired"),
    ("sie ist freundlich", "she is kind"),
    ("ich gehe jetzt", "i go now"),
    ("wir kommen morgen", "we come tomorrow"),
    ("ich verstehe das", "i understand that"),
    ("er lernt schnell", "he learns fast"),
    ("sie spricht englisch", "she speaks english"),
    ("ich schreibe einen brief", "i write a letter"),
    ("wir spielen zusammen", "we play together"),
    ("ich öffne die tür", "i open the door"),
    ("er schließt das fenster", "he closes the window"),
    ("ich höre musik", "i listen to music"),
    ("sie tanzt gut", "she dances well"),
    ("wir reisen viel", "we travel a lot"),
    ("ich koche gern", "i like cooking"),
    ("er liest viel", "he reads a lot"),
    ("sie arbeitet hart", "she works hard"),
    ("ich esse brot", "i eat bread"),
    ("wir trinken tee", "we drink tea"),
    ("ich schlafe jetzt", "i sleep now"),
]


In [2]:
# 2. Vocabulary
from collections import Counter

def build_vocab(sentences):
    words = []
    for s in sentences:
        words.extend(s.split())

    vocab = {"<pad>":0, "<sos>":1, "<eos>":2, "<unk>":3}

    for w in Counter(words):
        vocab[w] = len(vocab)

    return vocab

src_vocab = build_vocab([x[0] for x in data])  # German
tgt_vocab = build_vocab([x[1] for x in data])  # English

inv_tgt_vocab = {v:k for k,v in tgt_vocab.items()}


### Example of `build_vocab` function

In [27]:
sample_sentences = [
    "hello world",
    "world hello again"
]
sample_vocab = build_vocab(sample_sentences)
print(sample_vocab)

{'<pad>': 0, '<sos>': 1, '<eos>': 2, '<unk>': 3, 'hello': 4, 'world': 5, 'again': 6}


In [5]:
# 3. Encoding
def encode(sentence, vocab):
    return [vocab.get(w, vocab["<unk>"]) for w in sentence.split()]

def prepare_data(data):
    src_data, tgt_data = [], []

    for src, tgt in data:
        src_tensor = torch.tensor(encode(src, src_vocab))

        tgt_seq = [tgt_vocab["<sos>"]] + encode(tgt, tgt_vocab) + [tgt_vocab["<eos>"]]
        tgt_tensor = torch.tensor(tgt_seq)

        src_data.append(src_tensor)
        tgt_data.append(tgt_tensor)

    return src_data, tgt_data

src_data, tgt_data = prepare_data(data)


### Example of Encoded Data

Here are the first few encoded source and target sequences after running `prepare_data`:

In [28]:
print("First 3 source encoded sequences:")
for i in range(3):
    print(f"  Original German: '{data[i][0]}' -> Encoded: {src_data[i]}")

print("\nFirst 3 target encoded sequences:")
for i in range(3):
    # Decode target for better understanding
    decoded_tgt = ' '.join([inv_tgt_vocab[idx.item()] for idx in tgt_data[i]])
    print(f"  Original English: '{data[i][1]}' -> Encoded: {tgt_data[i]} -> Decoded: '{decoded_tgt}'")

First 3 source encoded sequences:
  Original German: 'ich bin ein student' -> Encoded: tensor([4, 5, 6, 7])
  Original German: 'ich liebe ki' -> Encoded: tensor([4, 8, 9])
  Original German: 'er ist klug' -> Encoded: tensor([10, 11, 12])

First 3 target encoded sequences:
  Original English: 'i am a student' -> Encoded: tensor([1, 4, 5, 6, 7, 2]) -> Decoded: '<sos> i am a student <eos>'
  Original English: 'i love ai' -> Encoded: tensor([1, 4, 8, 9, 2]) -> Decoded: '<sos> i love ai <eos>'
  Original English: 'he is smart' -> Encoded: tensor([ 1, 10, 11, 12,  2]) -> Decoded: '<sos> he is smart <eos>'


In [16]:
# 4. Model

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)

        for pos in range(max_len):
            for i in range(0, d_model, 2):
                pe[pos, i] = torch.sin(torch.tensor(pos / (10000 ** (i/d_model))))
                pe[pos, i+1] = torch.cos(torch.tensor(pos / (10000 ** (i/d_model))))

        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, heads):
        super().__init__()
        self.d_model = d_model
        self.heads = heads
        self.head_dim = d_model // heads

        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)
        self.fc = nn.Linear(d_model, d_model)

    def forward(self, query, key, value, mask=None):
        N = query.shape[0]

        Q = self.q(query)
        K = self.k(key)
        V = self.v(value)

        # Split heads
        Q = Q.view(N, -1, self.heads, self.head_dim).transpose(1,2)
        K = K.view(N, -1, self.heads, self.head_dim).transpose(1,2)
        V = V.view(N, -1, self.heads, self.head_dim).transpose(1,2)

        scores = torch.matmul(Q, K.transpose(-2,-1)) / (self.head_dim ** 0.5)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))

        attention = torch.softmax(scores, dim=-1)

        out = torch.matmul(attention, V)
        out = out.transpose(1,2).contiguous().view(N, -1, self.d_model)

        return self.fc(out)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, heads, forward_expansion=4):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_model*forward_expansion),
            nn.ReLU(),
            nn.Linear(d_model*forward_expansion, d_model)
        )

    def forward(self, x, mask=None):
        attn = self.attention(x, x, x, mask)
        x = self.norm1(attn + x)
        forward = self.feed_forward(x)
        return self.norm2(forward + x)

class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, heads, num_layers):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos = PositionalEncoding(d_model)

        self.layers = nn.ModuleList([
            TransformerBlock(d_model, heads) for _ in range(num_layers)
        ])

    def forward(self, x, mask=None):
        x = self.embed(x)
        x = self.pos(x)

        for layer in self.layers:
            x = layer(x, mask)

        return x

class DecoderBlock(nn.Module):
    def __init__(self, d_model, heads):
        super().__init__()

        self.self_attention = MultiHeadAttention(d_model, heads)
        self.norm1 = nn.LayerNorm(d_model)

        self.cross_attention = MultiHeadAttention(d_model, heads)
        self.norm2 = nn.LayerNorm(d_model)

        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_model*4),
            nn.ReLU(),
            nn.Linear(d_model*4, d_model)
        )
        self.norm3 = nn.LayerNorm(d_model)

    def forward(self, x, enc_out, src_mask, tgt_mask):

        # 1. Self Attention (on target)
        _x = self.self_attention(x, x, x, tgt_mask)
        x = self.norm1(_x + x)

        # 2. Cross Attention (VERY IMPORTANT)
        _x = self.cross_attention(x, enc_out, enc_out, src_mask)
        x = self.norm2(_x + x)

        # 3. Feed Forward
        _x = self.feed_forward(x)
        x = self.norm3(_x + x)

        return x


class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, heads, num_layers):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos = PositionalEncoding(d_model)

        self.layers = nn.ModuleList([
            DecoderBlock(d_model, heads) for _ in range(num_layers)
        ])

        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, x, enc_out, src_mask, tgt_mask):
        x = self.embed(x)
        x = self.pos(x)

        for layer in self.layers:
            x = layer(x, enc_out, src_mask, tgt_mask)

        return self.fc(x)



In [17]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=128, heads=4, num_layers=2):
        super().__init__()
        self.encoder = Encoder(src_vocab_size, d_model, heads, num_layers)
        self.decoder = Decoder(tgt_vocab_size, d_model, heads, num_layers)

    def forward(self, src, tgt):
        enc_out = self.encoder(src)
        out = self.decoder(tgt, enc_out, None, None)
        return out


In [24]:
# 5. Training : Took
model = Transformer(len(src_vocab), len(tgt_vocab)).to(device)


criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(100):  # increased a bit
    total_loss = 0

    for src, tgt in zip(src_data, tgt_data):
        src = src.unsqueeze(0).to(device)
        tgt = tgt.unsqueeze(0).to(device)

        output = model(src, tgt[:, :-1])

        output = output.reshape(-1, output.shape[-1])
        target = tgt[:, 1:].reshape(-1)

        loss = criterion(output, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss:.4f}")


Epoch 0, Loss: 143.4371
Epoch 20, Loss: 0.2669
Epoch 40, Loss: 0.0849
Epoch 60, Loss: 0.0391
Epoch 80, Loss: 0.0209


In [25]:
# 6. Translation
def translate(model, sentence):
    model.eval()

    src = torch.tensor(encode(sentence, src_vocab)).unsqueeze(0).to(device)
    enc_out = model.encoder(src)

    tgt = torch.tensor([[tgt_vocab["<sos>"]]]).to(device)

    for _ in range(10):
        out = model.decoder(tgt, enc_out, None, None)
        next_word = out.argmax(-1)[:, -1].item()

        tgt = torch.cat([tgt, torch.tensor([[next_word]]).to(device)], dim=1)

        if next_word == tgt_vocab["<eos>"]:
            break

    return " ".join([inv_tgt_vocab[i] for i in tgt[0].tolist()])


In [26]:
# 7. Test
print(translate(model, "ich bin ein student"))
print(translate(model, "ich liebe ki"))
print(translate(model, "sie ist glücklich"))


<sos> i <eos>
<sos> i <eos>
<sos> she is <eos>



“Why is this not perfect?”

Answer:
```
Tokenization problem
Small dataset problem
No masking
No batching